# Week 2 Day 1 - Which wording, and what does a mistake cost?

Burak Gulbol's take on the Week 2 Day 1 assignment: run the payout-tool agent on the concise policy wording, compare it with the full wording in Langfuse, then change the cost of a wrongful decline and see whether the choice changes.

One line in the handlers' notes made me look closer. CLM-1214 tests a rule "the concise wording leaves out": a delay on the way home keeps the cover going past the return date. The concise wording still says anything outside the trip dates is never covered. So the concise wording could turn a payout into a wrongful decline, the most expensive mistake.

| Experiment | Wording | Model |
|---|---|---|
| `full` | Full policy wording | GPT-6 Luna |
| `concise` | `policy_wording_short.md` | GPT-6 Luna |
| `concise_fixed` | Concise wording with the missing rule put back (one sentence) | GPT-6 Luna |
| `<best>_deepseek` | Whichever wording wins, on DeepSeek V4.1 Flash | DeepSeek V4.1 Flash |

Every experiment uses the payout tool, runs 3 times, and lands in Langfuse as its own dataset run.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

LABS = Path("../../labs")
sys.path.append(str(LABS))
from week2day1_payout import Benefit, Line, Payout, work_out_payout  # noqa: E402

In [ ]:
load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

langfuse = get_client()
print("Connected to Langfuse" if langfuse.auth_check() else "No Langfuse keys found")

TINY = False            # True runs the 5 tiny claims
N_TIMES = 3             # runs per experiment
MAX_CONCURRENCY = 4     # claims at once; set 1 if you see errors
RESULTS_FILE = Path("week2day1_results.csv")

## The harder claims and the two wordings

In [ ]:
DATA = Path("../../data")
HARDER = DATA / "evals" / "harder"


def load(path):
    return json.loads(path.read_text(encoding="utf-8"))


claims = load(HARDER / "claims.json")
labels = {d["claim_id"]: d for d in load(HARDER / "decisions.json")}
policies = {p["policy_id"]: p for p in load(HARDER / "policies.json")}

full_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
concise_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")
print(f"Full wording: {len(full_wording.split()):,} words. Concise: {len(concise_wording.split()):,} words")

### Putting the missing rule back

The full wording's Trip Delay section ends with the rule; the concise one doesn't have it. I add one sentence, in the concise style, at the end of the concise Trip Delay section. That's the whole fix: about 20 words.

In [ ]:
MISSING_RULE = "If a delay on the way home keeps the traveler away past the return date, cover continues until they get home."
assert "your cover continues until you get home" in full_wording
assert "continues until" not in concise_wording

anchor = "Needs the receipts and the carrier's confirmation of the delay and its cause."
assert concise_wording.count(anchor) == 1
concise_fixed_wording = concise_wording.replace(anchor, f"{anchor} {MISSING_RULE}")

WORDINGS = {"full": full_wording, "concise": concise_wording, "concise_fixed": concise_fixed_wording}

The lab's instructions say "work out the payout using section 10". The concise wording has no numbered sections, so for it the instructions name its Payment section. Otherwise the instructions are the lab's.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using {payment}.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
Use the payout tool to work out the payout on an approved claim, and record the payout it returns.
"""


def system_prompt(wording_name):
    payment = "section 10" if wording_name == "full" else "the Payment section"
    return f"{INSTRUCTIONS.format(payment=payment)}\n{WORDINGS[wording_name]}\n"


def make_user_prompt(claim):
    policy = policies[claim["policy_id"]]
    return f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"

## The tools, as in the lab

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

In [ ]:
MODELS = {
    "luna": dict(model="openai/gpt-6-luna", reasoning_effort="none"),
    "deepseek": dict(model="deepseek/deepseek-v4.1-flash", reasoning_effort="low"),
}


def make_agent(wording_name, model_name):
    llm = ChatOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **MODELS[model_name])
    return create_agent(model=llm, tools=[record_decision, calculate_payout], system_prompt=system_prompt(wording_name))

## Scoring

The lab's scoring, with one change: `mistake_cost` takes the cost of a wrongful decline as an argument, so the same answers can be scored again with a different figure without calling a model.

I also record each claim's time and input tokens, to answer "is it cheaper or faster?".

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected, wrongful_decline=WRONGFUL_DECLINE):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return wrongful_decline
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def kappa(table):
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(d, 0) * handler_share.get(d, 0) for d in DECISIONS)
    return (observed - expected) / (1 - expected)


def run_scores(table):
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "wrongful_declines": ((table["agent"] == "decline") & (table["handler"] != "decline")).sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
        "seconds_a_claim": table["seconds"].mean(),
        "input_tokens_a_claim": table["input_tokens"].mean(),
    }


def score_claim(*, output, expected_output, **kwargs):
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"], "agent": answer["decision"],
            "handler_payout": expected["payout"], "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"], "seconds": answer["seconds"], "input_tokens": answer["input_tokens"],
            "rationale": answer["rationale"],
            "trace_id": result.trace_id, "dataset_run_id": result.dataset_run_id,
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def score_run(*, item_results, **kwargs):
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

## Running the experiments

In [ ]:
langfuse_handler = CallbackHandler()
dataset = langfuse.get_dataset("harder-claims")   # made in the lab's Part 4
items = [i for i in dataset.items if i.metadata["tiny"]] if TINY else dataset.items
print(f"{len(items)} claims")


def decide(agent, claim):
    start = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    usage = [m.response_metadata.get("token_usage", {}) for m in result["messages"] if m.type == "ai"]
    return {
        **answer,
        "cost": sum(u.get("cost") or 0 for u in usage),
        "input_tokens": sum(u.get("prompt_tokens") or 0 for u in usage),
        "seconds": round(time.time() - start, 1),
    }


def experiment(wording_name, model_name="luna"):
    """Run one experiment N_TIMES in Langfuse, saving every run's answers to RESULTS_FILE as it finishes."""
    name = wording_name if model_name == "luna" else f"{wording_name}_{model_name}"
    agent = make_agent(wording_name, model_name)
    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        result = langfuse.run_experiment(
            name=name,
            run_name=f"{name} run {run}",
            description=f"Payout tool, {wording_name} wording, {MODELS[model_name]['model']}",
            data=items,
            task=lambda *, item, **kwargs: decide(agent, item.input),
            evaluators=[score_claim],
            run_evaluators=[score_run],
            max_concurrency=MAX_CONCURRENCY,
        )
        table = score_table(result.item_results).assign(experiment=name, run=run)
        table.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
        tables.append(table)
        print(f"{name} run {run}: {table['right'].sum()} of {len(table)} right, "
              f"{table['mistake_cost'].sum():,.0f} dollars of mistakes, {table['seconds'].mean():.1f} s a claim")
    return pd.concat(tables, ignore_index=True)

In [ ]:
full = experiment("full")

In [ ]:
concise = experiment("concise")

In [ ]:
concise_fixed = experiment("concise_fixed")

## Results

In [ ]:
def average_scores(table):
    return pd.DataFrame([run_scores(t) for _, t in table.groupby("run")]).mean()


results = pd.concat([full, concise, concise_fixed], ignore_index=True)
summary = pd.DataFrame({name: average_scores(t) for name, t in results.groupby("experiment")}).T
summary.round(3)

### Which claims the wordings disagree on

How many of the runs got each claim right, for the claims where the experiments differ. The handlers' note says what each claim tests.

In [ ]:
by_claim = results.pivot_table(index="claim_id", columns="experiment", values="right", aggfunc="sum")
by_claim["handler"] = [f"{labels[c]['decision']} {labels[c]['payout']:,.2f}" for c in by_claim.index]
by_claim["tests"] = [labels[c]["tests"][:90] for c in by_claim.index]
pd.set_option("display.max_colwidth", 120)
by_claim[by_claim.drop(columns=["handler", "tests"]).nunique(axis=1) > 1]

In [ ]:
# What each experiment did on CLM-1214, the claim the concise wording was expected to get wrong
results[results.claim_id == "CLM-1214"][["experiment", "run", "agent", "agent_payout", "handler_payout", "mistake_cost", "rationale"]]

## Changing what a wrongful decline costs

The lab puts a wrongful decline at 1,500 dollars. Here the same answers are scored again at 500, 1,500, 3,000 and 5,000, without calling the model again. If the lines cross, the cheapest wording depends on a business figure, not on the model.

In [ ]:
WRONGFUL_DECLINE_VALUES = [500, 1500, 3000, 5000]


def cost_of_mistakes(table, wrongful_decline):
    """The cost of mistakes per run, averaged over the runs, at a given cost of a wrongful decline."""
    costs = table.apply(
        lambda r: mistake_cost({"decision": r.agent, "payout": r.agent_payout},
                               {"decision": r.handler, "payout": r.handler_payout}, wrongful_decline),
        axis=1,
    )
    return costs.groupby(table["run"]).sum().mean()


sensitivity = pd.DataFrame([
    {"experiment": name, "wrongful_decline": value, "cost_of_mistakes": cost_of_mistakes(t, value)}
    for name, t in results.groupby("experiment") for value in WRONGFUL_DECLINE_VALUES
])
fig = px.line(sensitivity, x="wrongful_decline", y="cost_of_mistakes", color="experiment", markers=True,
              title="Cost of mistakes per run of 25 claims, against the cost of a wrongful decline", height=500)
fig.show()
sensitivity.pivot(index="experiment", columns="wrongful_decline", values="cost_of_mistakes").round(0)

### The new scores in Langfuse

So the comparison in Langfuse matches, each dataset run gets a run-level score for each figure, named `cost_of_mistakes_wd<figure>`. In Langfuse, open Datasets, then `harder-claims`, and add these scores as columns when you compare runs.

In [ ]:
for (name, run), table in results.groupby(["experiment", "run"]):
    run_id = table["dataset_run_id"].iloc[0]
    for value in WRONGFUL_DECLINE_VALUES:
        langfuse.create_score(name=f"cost_of_mistakes_wd{value}", value=float(cost_of_mistakes(table.assign(run=1), value)),
                              dataset_run_id=run_id)
langfuse.flush()
print("Scores added")

## The same choice on a cheaper model

DeepSeek V4.1 Flash at low reasoning was the best balance on Week 1 Day 4. Set `BEST` to the wording that won above and run it.

In [ ]:
BEST = "concise_fixed"
deepseek = experiment(BEST, "deepseek")
results = pd.concat([results, deepseek], ignore_index=True)
summary = pd.DataFrame({name: average_scores(t) for name, t in results.groupby("experiment")}).T
summary.round(3)

In [ ]:
fig = px.scatter(summary.reset_index(names="experiment"), x="llm_cost", y="cost_of_mistakes", text="experiment",
                 title="Cost of mistakes against LLM cost, per run of 25 claims (bottom left is best)", height=500)
fig.update_traces(marker_size=14, textposition="top center")
fig.show()

## One more round: the example the concise wording dropped

The handlers' note pointed at CLM-1214, but every wording got CLM-1214 right in all 9 runs. The claim the concise wording actually broke is CLM-1216: right 3 of 3 with the full wording, 0 of 3 with the concise one, and 0 of 3 on DeepSeek too. Each time it paid 2,005.90 dollars where the handler paid 1,677.90.

The full wording says "A pair or a set, **such as a pair of shoes, a suit or a camera with its lens**, counts as one item." The concise wording keeps the rule but drops the examples, so the model counts the camera and the lens as two items, each with its own limit.

So: put the examples back (eleven words) and measure again.

If the kernel was restarted, run the cells from the top down to the one that defines `experiment`, skip the experiment cells, and carry on here. The comparison below reads every run from `week2day1_results.csv`, so nothing needs running twice.

In [ ]:
def average_scores(table):
    return pd.DataFrame([run_scores(t) for _, t in table.groupby("run")]).mean()


WRONGFUL_DECLINE_VALUES = [500, 1500, 3000, 5000]


def cost_of_mistakes(table, wrongful_decline):
    costs = table.apply(
        lambda r: mistake_cost({"decision": r.agent, "payout": r.agent_payout},
                               {"decision": r.handler, "payout": r.handler_payout}, wrongful_decline),
        axis=1,
    )
    return costs.groupby(table["run"]).sum().mean()

In [ ]:
EXAMPLES = "a pair or a set counts as one item"
assert concise_fixed_wording.count(EXAMPLES) == 1
WORDINGS["concise_example"] = concise_fixed_wording.replace(
    EXAMPLES, "a pair or a set, such as a pair of shoes, a suit or a camera with its lens, counts as one item"
)
print(f"concise_example: {len(WORDINGS['concise_example'].split()):,} words")

In [ ]:
concise_example = experiment("concise_example")

### Every run so far, from the results file

In [ ]:
all_results = pd.read_csv(RESULTS_FILE).drop_duplicates(subset=["experiment", "run", "claim_id"], keep="last")
ORDER = ["full", "concise", "concise_fixed", "concise_example", "concise_fixed_deepseek"]
summary = pd.DataFrame({name: average_scores(t) for name, t in all_results.groupby("experiment")}).T.reindex(ORDER).dropna(how="all")
summary.round(3)

In [ ]:
# CLM-1216 in every experiment: the right payout is 1,677.90
claim_1216 = all_results[all_results.claim_id == "CLM-1216"]
claim_1216.pivot_table(index="experiment", columns="run", values="agent_payout").reindex(ORDER).dropna(how="all")

In [ ]:
sensitivity = pd.DataFrame([
    {"experiment": name, "wrongful_decline": value, "cost_of_mistakes": cost_of_mistakes(t, value)}
    for name, t in all_results.groupby("experiment") for value in WRONGFUL_DECLINE_VALUES
])
fig = px.line(sensitivity, x="wrongful_decline", y="cost_of_mistakes", color="experiment", markers=True,
              title="Cost of mistakes per run of 25 claims, against the cost of a wrongful decline", height=500)
fig.show()
sensitivity.pivot(index="experiment", columns="wrongful_decline", values="cost_of_mistakes").reindex(ORDER).dropna(how="all").round(0)

In [ ]:
# The new run's scores in Langfuse too
for run, table in concise_example.groupby("run"):
    for value in WRONGFUL_DECLINE_VALUES:
        langfuse.create_score(name=f"cost_of_mistakes_wd{value}", value=float(cost_of_mistakes(table, value)),
                              dataset_run_id=table["dataset_run_id"].iloc[0])
langfuse.flush()
print("Scores added")